# BITS Pilani - M.Tech (DSE) Machine Learning
## Assignment 2: Classification Model Benchmark & Deployment

**Dataset**: Online Shoppers Purchasing Intention (UCI Machine Learning Repository)
**Target Variable**:  (1 = Completed Purchase, 0 = No Purchase)
**Models Implemented**:
1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbors (kNN)
4. Naive Bayes Classifier (GaussianNB)
5. Support Vector Machine (SVM)

In [ ]:
import os
import io
import json
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    classification_report, confusion_matrix
)

### 1. Dataset Acquisition & Verification
We load the Online Shoppers Purchasing Intention Dataset from the UCI Machine Learning Repository.

In [ ]:
# Load dataset from local test_data.csv or fetch from UCI
test_csv_path = '../test_data.csv'
if os.path.exists(test_csv_path):
    print(f'Test data exists locally at {test_csv_path}')


### 2. Preprocessing & Feature Engineering
We construct a  that standardizes numerical features and one-hot encodes categorical attributes.

In [ ]:
numerical_cols = [
    'Administrative', 'Administrative_Duration', 
    'Informational', 'Informational_Duration', 
    'ProductRelated', 'ProductRelated_Duration', 
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]
categorical_cols = [
    'Month', 'OperatingSystems', 'Browser', 
    'Region', 'TrafficType', 'VisitorType', 'Weekend'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

### 3. Model Benchmark & Evaluation
We evaluate all 5 models using the saved artifacts on test_data.csv.

In [ ]:
test_df = pd.read_csv('../test_data.csv')
X_test = test_df.drop(columns=['Revenue'])
y_test = test_df['Revenue']

model_names = [
    'Logistic Regression',
    'Decision Tree',
    'kNN',
    'Naive Bayes',
    'Support Vector Machine'
]

file_map = {
    'Logistic Regression': 'logistic_regression.joblib',
    'Decision Tree': 'decision_tree.joblib',
    'kNN': 'knn.joblib',
    'Naive Bayes': 'naive_bayes.joblib',
    'Support Vector Machine': 'svm.joblib'
}

results = []
for name in model_names:
    model_path = file_map[name]
    pipeline = joblib.load(model_path)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, 'predict_proba') else y_pred
    
    results.append({
        'ML Model Name': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_prob),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_test, y_pred)
    })

comparison_df = pd.DataFrame(results).set_index('ML Model Name')
display(comparison_df.round(4))
